# 📉 Cuaderno 01 — Derivación numérica

**Métodos Numéricos · Unidad 5: Derivadas, integrales y EDO**

Tema **5.1 Derivación numérica** (capítulo 23 del libro).

En este cuaderno **no programas** el método: aprendes a **usar** las
funciones nativas de NumPy (`np.gradient`, `np.diff`) y su equivalente en
MATLAB (`gradient`, `diff`), entendiendo qué calculan, qué reciben, qué
devuelven y cuándo conviene cada una.

> Camino del cuaderno: **teoría → laboratorio → entradas/salidas →
> preguntas → caso de ingeniería → envío**.

In [ ]:
#@title ⚙️ Entorno — ejecuta esta celda PRIMERO
import os, sys, json

CUADERNO = 'u5_01_derivacion'
TAREA_ID = 'U5_01_Derivacion_Numerica'
REPO_NOMBRE = 'U5 derivadas integrales y EDO'

def _raiz_local():
    b = os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(b, 'lib', 'robomat.py')):
            return b
        padre = os.path.dirname(b)
        if padre == b:
            break
        b = padre
    return None

try:
    import google.colab  # noqa: F401
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

if ES_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = os.path.join('/content/drive/MyDrive', 'CursoMN', REPO_NOMBRE)
    if not os.path.exists(REPO):
        raise RuntimeError('Clona tu fork en MyDrive/CursoMN/' + REPO_NOMBRE)
else:
    REPO = _raiz_local()
    if REPO is None:
        raise RuntimeError('Abre el cuaderno dentro de ' + REPO_NOMBRE)

sys.path.insert(0, os.path.join(REPO, 'lib'))
print('Modo       :', 'Colab' if ES_COLAB else 'local')
print('Repositorio:', REPO)

In [ ]:
#@title 📦 Librerías
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate

# Preludio estilo MATLAB (opcional, para mostrar matrices/tablas)
exec(open(os.path.join(REPO, 'lib', 'robomat.py'), encoding='utf-8').read(), globals())

print('NumPy  ', np.__version__)
print('SciPy  ', __import__('scipy').__version__)

## 1. Teoría: ¿qué es la derivación numérica?

La **derivada** mide la razón de cambio de $f(x)$ respecto a $x$ y, en la
práctica, a menudo solo tenemos **datos discretos** $(x_i, f(x_i))$, no la
función continua. La derivación numérica aproxima $f'(x)$ usando
**diferencias finitas** entre puntos vecinos.

Las tres familias básicas (figuras 23.1–23.3 del libro) son:

**Hacia adelante** (usa $x_i$ y $x_{i+1}$):

$$f'(x_i) = \frac{f(x_{i+1}) - f(x_i)}{h} \quad O(h)$$

**Hacia atrás** (usa $x_i$ y $x_{i-1}$):

$$f'(x_i) = \frac{f(x_i) - f(x_{i-1})}{h} \quad O(h)$$

**Centrada** (usa $x_{i-1}$ y $x_{i+1}$, más exacta):

$$f'(x_i) = \frac{f(x_{i+1}) - f(x_{i-1})}{2h} \quad O(h^2)$$

donde $h$ es el espaciado entre puntos. La notación $O(h)$ / $O(h^2)$ dice
**qué tan rápido baja el error** al reducir $h$: centrada ($O(h^2)$) mejora
al cuadrado, por eso es la preferida cuando se puede usar (puntos a ambos
lados).

> 📖 **Figuras del libro** (para la teoría visual):
> `P6_23_Diferenciacion_numerica_images/vector/fig_23_4.pdf` (temperatura vs
> profundidad, ejemplo de datos tabulados). Las tablas de fórmulas son las
> **figuras 23.1 (adelante), 23.2 (atrás) y 23.3 (centrada)**.

**Referencia por dentro:** en `lib/derivada_integral.py` están
`derivada_adelante`, `derivada_atras` y `derivada_centrada`, que implementan
exactamente estas fórmulas. Las vas a ver, pero **tú usas NumPy**.

## 2. Laboratorio: derivar con NumPy

Dada la posición de un objeto $s(t) = t^3$ (metros) en función del tiempo
$t$ (segundos), la velocidad es $v(t) = s'(t) = 3t^2$. Vamos a calcularla
numéricamente con **funciones nativas** y comparar contra la solución exacta.

In [ ]:
# Datos tabulados: posición s(t) = t^3 en puntos equiespaciados
t = np.arange(0.0, 4.01, 0.5)        # 0, 0.5, 1.0, ..., 4.0
s = t ** 3

# --- Derivación numérica con funciones nativas ---
v_centrada = np.gradient(s, t)        # centrada O(h^2) en el interior
v_adelante = np.diff(s) / np.diff(t)  # hacia adelante O(h), un punto menos

# Solución exacta v(t) = 3 t^2
v_exacta = 3.0 * t ** 2

print('   t   | v exacta | v centrada | error centrada')
for i in range(len(t)):
    print(f'{t[i]:5.1f} | {v_exacta[i]:8.4f} | {v_centrada[i]:10.4f} | '
          f'{abs(v_centrada[i]-v_exacta[i]):13.2e}')

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(t, v_exacta, 'k-', label='exacta $v(t)=3t^2$')
plt.plot(t, v_centrada, 'ro', label='NumPy `np.gradient` (centrada)')
plt.plot(t[:-1], v_adelante, 'bs', label='NumPy `np.diff` (adelante)')
plt.xlabel('t (s)'); plt.ylabel('v (m/s)')
plt.legend(); plt.grid(alpha=0.3); plt.title('Derivación numérica de s(t)=t³')
plt.show()

## 3. Entradas, salidas y cuándo usar cada una

### `np.gradient(y, x)`

- **Entrada:** `y` (valores de la función) y `x` (abscisas, **opcional**; si
  se omite, asume espaciado 1).
- **Salida:** arreglo del **mismo tamaño** que `y` con la derivada en cada
  punto.
- **Cómo calcula:** diferencias **centradas** $O(h^2)$ en el interior, y
  **adelante/atrás** $O(h)$ en los extremos (donde no hay punto a ambos
  lados).
- **Cuándo usarla:** derivada de datos tabulados, espaciados igual o
  irregular; es la opción por omisión para **una derivada suave**.

### `np.diff(y) / np.diff(x)`

- **Entrada:** `y` y `x` (arreglos del mismo tamaño).
- **Salida:** arreglo de **tamaño `n-1`** (una diferencia por cada par
  consecutivo).
- **Cómo calcula:** diferencia **hacia adelante** $O(h)$: es la pendiente de
  cada segmento.
- **Cuándo usarla:** cuando solo quieres la pendiente *entre* puntos, o en el
  borde donde no hay punto siguiente/previo.

> 🐍 **Equivalente MATLAB:** `gradient(y, h)` ≈ `np.gradient` y
> `diff(y)./diff(x)` ≈ `np.diff(y)/np.diff(x)`.

**Diferencia clave:** `np.gradient` devuelve una derivada **por punto** (con
centrada en el interior) y es más exacta; `np.diff` devuelve una pendiente
**por intervalo** (adelante) y pierde un punto.

## 4. Pregunta teórica

**P1.** ¿Por qué la diferencia centrada
$f'(x_i) = \frac{f(x_{i+1}) - f(x_{i-1})}{2h}$ es más exacta que la diferencia
hacia adelante $\frac{f(x_{i+1}) - f(x_i)}{h}$?

1. Porque usa un espaciado $2h$ más grande.
2. Porque su error es $O(h^2)$, mientras que el de adelante es $O(h)$.
3. Porque solo requiere un punto, no dos.
4. Porque es la única que funciona con datos irregulares.

In [ ]:
# Responde aquí con el número de la opción correcta
RESPUESTA_P1 = 2

## 5. Caso de ingeniería: gradiente de temperatura

La **ley de Fourier** del calor dice que el flujo de calor es proporcional al
gradiente de temperatura: $q = -k\, dT/dx$. Un sensor mide la temperatura del
suelo a distintas profundidades; queremos estimar **dónde cambia más rápido**
la temperatura.

| Profundidad $x$ (m) | 0 | 0.5 | 1.0 | 1.5 | 2.0 |
|---|---|---|---|---|---|
| Temperatura $T$ (°C) | 25 | 22 | 19 | 17 | 15 |

Usa `np.gradient` para obtener $dT/dx$ y responde: ¿en qué zona el gradiente
es más negativo (mayor enfriamiento por metro)?

In [ ]:
x = np.array([0.0, 0.5, 1.0, 1.5, 2.0])
T = np.array([25.0, 22.0, 19.0, 17.0, 15.0])

dTdx = np.gradient(T, x)   # gradiente de temperatura por metro

print(' x (m) |  T (°C) | dT/dx (°C/m)')
for xi, Ti, di in zip(x, T, dTdx):
    print(f'{xi:5.1f}  | {Ti:7.1f}  | {di:10.2f}')

i_min = int(np.argmin(dTdx))
print(f'\nGradiente más negativo en x = {x[i_min]} m (dT/dx = {dTdx[i_min]:.2f} °C/m)')

# Respuesta al caso (para autoevaluar):
RESPUESTA_CASO = float(dTdx[i_min])

## 6. Revisar, enviar y ver calificación

Las respuestas (`RESPUESTA_P1`, `RESPUESTA_CASO`) se envían al **Apps Script**
para calificarse. Esta sección se conecta al backend en la Fase 4 (motor +
blob en `grader/`).

In [ ]:
#@title 📤 Enviar respuestas y ver calificación
# (Se activa al conectar el motor y el Apps Script en la Fase 4.)
alumno_id = 'TU_NUMERO_CONTROL'
respuestas = {'P1': RESPUESTA_P1, 'CASO': RESPUESTA_CASO}
print('Pendiente de conectar con Apps Script:', respuestas)